In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('/content/Final_Augmented_dataset_Diseases_and_Symptoms.csv')

In [ ]:
df.shape

(246945, 378)

In [ ]:
df.isnull().sum()

,0
diseases,0
anxiety and nervousness,0
depression,0
shortness of breath,0
depressive or psychotic symptoms,0
...,...
hip weakness,0
back swelling,0
ankle stiffness or tightness,0
ankle weakness,0


In [ ]:
print(df['diseases'].nunique())
print(df['diseases'].value_counts().tail(20))   # check the rarest classes

773
diseases
open wound of the jaw           2
turner syndrome                 1
rocky mountain spotted fever    1
open wound of the cheek         1
high blood pressure             1
open wound due to trauma        1
open wound of the chest         1
huntington disease              1
open wound of the knee          1
foreign body in the nose        1
diabetes                        1
thalassemia                     1
heat stroke                     1
gas gangrene                    1
typhoid fever                   1
open wound of the head          1
myocarditis                     1
chronic ulcer                   1
hypergammaglobulinemia          1
kaposi sarcoma                  1
Name: count, dtype: int64


In [ ]:
symptom_cols = df.columns.drop('diseases')
df[symptom_cols] = df[symptom_cols].astype('int8')   # 8x smaller than float64
print(df.memory_usage(deep=True).sum() / 1e6, "MB after downcast")


109.535362 MB after downcast


In [ ]:
USE_SAMPLE = True
if USE_SAMPLE:
    df = df.sample(n=100000, random_state=42).reset_index(drop=True)
    print("Subsampled to:", df.shape)

Subsampled to: (100000, 378)


In [ ]:
counts = df['diseases'].value_counts()  #Handle rare/singleton classes
print((counts == 1).sum(), "diseases with only 1 row (will be dropped)")


24 diseases with only 1 row (will be dropped)


In [ ]:
MIN_SAMPLES = 2
valid_diseases = counts[counts >= MIN_SAMPLES].index
df = df[df['diseases'].isin(valid_diseases)].reset_index(drop=True)

In [ ]:
print("After filtering:", df.shape)
print( df['diseases'].nunique(), "diseases remain")


After filtering: (99976, 378)
726 diseases remain


In [ ]:
X = df.drop(columns=['diseases'])
y = df['diseases']


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import joblib


In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)


In [ ]:
import gc #Garbage Collection

# free the original dataframe to save memory, X,y already hold what we need
if 'df' in globals():
    del df
gc.collect() #Frees unused memory.

117

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,       # reduced from 200 - fewer trees = less RAM
    max_depth=25,
    min_samples_leaf=3,     # fewer nodes per tree
    n_jobs=2,               # limit parallel workers - each worker holds its own memory copy
    random_state=42,
    class_weight='balanced'
)
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=25,
                       min_samples_leaf=3, n_jobs=2, random_state=42)

In [ ]:
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.7414482896579316


In [ ]:
report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).transpose()
print("\nBest predicted diseases:")
print(report_df.sort_values('f1-score', ascending=False).head(15))
print("\nWorst predicted diseases:")
print(report_df.sort_values('f1-score', ascending=True).head(15))


Best predicted diseases:
     precision  recall  f1-score  support
696        1.0     1.0       1.0      1.0
29         1.0     1.0       1.0      4.0
58         1.0     1.0       1.0      5.0
676        1.0     1.0       1.0      3.0
661        1.0     1.0       1.0      2.0
655        1.0     1.0       1.0     13.0
567        1.0     1.0       1.0     16.0
610        1.0     1.0       1.0     18.0
652        1.0     1.0       1.0      1.0
653        1.0     1.0       1.0      4.0
577        1.0     1.0       1.0      2.0
496        1.0     1.0       1.0      6.0
477        1.0     1.0       1.0     13.0
516        1.0     1.0       1.0      5.0
522        1.0     1.0       1.0      5.0

Worst predicted diseases:
     precision  recall  f1-score  support
13         0.0     0.0       0.0      1.0
326        0.0     0.0       0.0      1.0
323        0.0     0.0       0.0      0.0
312        0.0     0.0       0.0      0.0
268        0.0     0.0       0.0      1.0
300        0.0     0.0 

In [ ]:
#Feature importance
importances = pd.Series(model.feature_importances_, index=X.columns)
top_symptoms = importances.sort_values(ascending=False).head(20)
print(top_symptoms)

emotional symptoms                  0.015373
shortness of breath                 0.015011
cough                               0.014431
headache                            0.013479
sharp abdominal pain                0.012437
elbow weakness                      0.012310
dizziness                           0.012039
sharp chest pain                    0.011912
depressive or psychotic symptoms    0.011286
knee lump or mass                   0.011247
vomiting                            0.010964
difficulty speaking                 0.010109
fatigue                             0.009426
anxiety and nervousness             0.009077
hot flashes                         0.008920
lack of growth                      0.008408
nausea                              0.008219
arm stiffness or tightness          0.007907
fainting                            0.007733
involuntary urination               0.007503
dtype: float64


In [ ]:
joblib.dump(model, 'disease_predictor_rf.joblib')
joblib.dump(le, 'disease_label_encoder.joblib')

['disease_label_encoder.joblib']

In [ ]:
def predict_disease(symptom_dict):

    row = pd.Series(0, index=X.columns)
    for symptom, val in symptom_dict.items():
        if symptom in row.index:
            row[symptom] = val
    pred_enc = model.predict([row.values])[0]
    disease = le.inverse_transform([pred_enc])[0]
    return disease

In [ ]:
result = predict_disease({'sharp chest pain': 1, 'dizziness': 1, 'palpitations': 1})
print("Predicted disease:", result)


Predicted disease: endocarditis


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
